# Uso del Sistema Multiagente

**Curso:** Probabilidad y Estadística Inferencial
**Institución:** Universidad de la Ciénega del Estado de Michoacán de Ocampo (UCEMICH)
**Carrera:** Ingeniería en Nanotecnología

Este notebook es material de referencia técnica, **no una unidad evaluada del curso**: vive en `notebooks_extra/` en vez de `lecciones/`, por lo que no pasa por el pipeline de compilación `.md -> .ipynb` ni por el gate de auditoría del Consejo de Expertos (`OrchestratorAgent`). Se edita directamente como notebook.

## Para quién es esto

- **Alumnos**: curiosidad de cómo funciona el sistema que audita y tutoriza este curso por dentro.
- **Profesores y colegas**: cómo invocar el pipeline de auditoría pedagógica sobre las lecciones reales, y cómo hacerle una pregunta puntual al tutor de IA.
- **Colegas técnicos**: la API real de los dos puntos de entrada de más alto nivel del sistema (`OrchestratorAgent`, `StatsTutorAgent`), con ejemplos ejecutables en vez de tener que leer los tests para inferir cómo se usan.

## Qué vas a ver

1. **`OrchestratorAgent`** — vista completa: corre el pipeline de auditoría de código, contenido pedagógico y el Consejo de 8 agentes sobre las unidades reales del curso, y cómo interpretar su reporte.
2. **`StatsTutorAgent`** — vista temática: cómo hacerle una pregunta puntual sobre un tema del curso (aquí, estimación de densidad de kernel / KDE) usando su RAG sobre lecciones y bibliografía.

## Requisitos antes de correr este notebook

- Entorno `ia_stats` activado (`conda activate ia_stats`), con las dependencias del repo instaladas.
- Para la sección 2 (`StatsTutorAgent`): un archivo `.env` en la raíz del repo con `GEMINI_API_KEY=tu_key` (nunca la pegues en una celda ni la subas a git -- `.env` ya está en `.gitignore`).

---

## 1. `OrchestratorAgent`: pipeline completo de auditoría pedagógica

`OrchestratorAgent` es el punto de entrada de más alto nivel del sistema: coordina la compilación de notebooks, la auditoría de código (PEP8/seguridad), la auditoría de contenido pedagógico y el Consejo de 8 agentes de gobernanza (`Safety Gate`, `Scientist`, `Analyst`, `Engineer`, `Editor`, `Architect`, `Librarian`, `QA`) sobre **todas** las unidades de `lecciones/*.md` a la vez -- no recibe un tema o unidad individual como parámetro, es un pipeline de compilación completo.

Instanciarlo y correrlo sobre el repo real:

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "multiagent_core").exists():
    # Si el notebook se corre desde notebooks_extra/, sube un nivel.
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.multiagent_core.orchestrator_agent import OrchestratorAgent

orquestador = OrchestratorAgent(
    lecciones_dir=str(REPO_ROOT / "lecciones"),
    notebooks_dir=str(REPO_ROOT / "notebooks"),
)

`run_full_pipeline(enforce_gate=True)` recorre cada `UNIDAD_*.md`, corre el Consejo como gate bloqueante, y solo compila a `.ipynb` las unidades que lo pasan. Con `enforce_gate=True` (el default), una unidad bloqueada no se compila -- eso es intencional: nunca se publica una unidad con un defecto real detectado por el Consejo.

Los reportes bloqueantes son `engineer`, `editor`, `scientist`, `analyst` y `safety_gate` (los únicos con lógica real de detección de un defecto de publicación, según `GOVERNANCE.md`); `architect`, `librarian` y `qa` son informativos/advisory.

In [2]:
reportes = orquestador.run_full_pipeline(enforce_gate=True)

print(f"Unidades procesadas: {len(reportes)}\n")
for reporte in reportes:
    unidad = reporte["md_filename"]
    bloqueada = reporte["gate_blocked"]
    estado = "🚫 bloqueada" if bloqueada else "✅ compilada"
    print(f"{unidad}: {estado}")

Unidades procesadas: 8

UNIDAD_1_ESTADISTICA_DESCRIPTIVA.md: ✅ compilada
UNIDAD_2_PROBABILIDAD_COMBINATORIA.md: ✅ compilada
UNIDAD_3_VARIABLES_ALEATORIAS_DISCRETAS.md: ✅ compilada
UNIDAD_4_DISTRIBUCIONES_CONJUNTAS.md: ✅ compilada
UNIDAD_5_VARIABLES_ALEATORIAS_CONTINUAS.md: ✅ compilada
UNIDAD_6_MODELADO_SIMULACION.md: ✅ compilada
UNIDAD_7_INFERENCIA_ESTIMACION.md: ✅ compilada
UNIDAD_8_PROYECTO_INTEGRADOR.md: ✅ compilada


Para inspeccionar el detalle de una unidad específica (por ejemplo, por qué el Consejo bloqueó o aprobó UNIDAD_1), se puede indexar el resultado:

In [3]:
reporte_u1 = next(r for r in reportes if "UNIDAD_1" in r["md_filename"])

print("Claves del reporte:", list(reporte_u1.keys()))
print()
print(f"Bloqueada por el gate: {reporte_u1['gate_blocked']}")
if reporte_u1["gate_reason"]:
    print(f"Motivo: {reporte_u1['gate_reason']}")
print()
print("Checklist del Hilo de Oro (ContentAuditorAgent):")
for componente, cumple in reporte_u1["content_audit"]["component_checks"].items():
    marca = "✅" if cumple else "❌"
    print(f"  {marca} {componente}")

Claves del reporte: ['md_filename', 'notebook_path', 'content_audit', 'code_audit', 'evaluation', 'approved', 'gate_blocked', 'gate_reason']

Bloqueada por el gate: False

Checklist del Hilo de Oro (ContentAuditorAgent):
  ✅ Teoría Completa
  ✅ Ejemplo Analítico
  ✅ Verificación SymPy
  ✅ Contexto Nanotecnológico
  ✅ Solución en \boxed{}
  ✅ Solución Computacional SciPy
  ✅ Visualización Profesional
  ✅ Interpretación Post-Gráfico
  ✅ Diccionario de Variables


---

## 2. `StatsTutorAgent`: preguntas temáticas puntuales

A diferencia del orquestador (que audita unidades completas), `StatsTutorAgent` responde preguntas puntuales sobre cualquier tema del curso, combinando:

- **RAG semántico sobre `lecciones/*.md`** (ChromaDB, colección `lecciones_probabilidad`).
- **RAG semántico sobre la bibliografía académica** (`bibliografia/*.pdf`, colección `bibliografia_pdfs`).
- **Gemini** (`gemini-2.5-flash`) para redactar la respuesta final en español, con contexto de nanotecnología.

Requiere `GEMINI_API_KEY` en el entorno (cargada aquí desde `.env` con `python-dotenv`, nunca pegada en una celda).

In [4]:
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from src.multiagent_core.stats_tutor_agent import StatsTutorAgent

tutor = StatsTutorAgent(course_dir=REPO_ROOT / "lecciones")

print(f"Colección lecciones: {tutor.collection.count()} documentos")
print(f"Colección bibliografía: {tutor.bibliografia_collection.count()} documentos")

chroma_path 'C:\Users\ljyud\Desktop\IA UCEMICH\PROBABILIDAD Y ESTADÍSTICA\lecciones\.chroma' contiene caracteres no-ASCII, lo que corrompe el índice HNSW en Windows. Redirigiendo a 'C:\Users\ljyud\AppData\Local\Temp\multiagent_core_chroma\217b8521e08c2d7f'. Este índice vive fuera de la carpeta del curso: bórralo manualmente ahí si necesitas reindexar desde cero.


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


C:\Users\ljyud\anaconda3\envs\ia_stats\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4250.65it/s]

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Colección lecciones: 374 documentos
Colección bibliografía: 12582 documentos


La primera instanciación indexa `lecciones/*.md` y `bibliografia/*.pdf` si aún no están indexados (puede tardar); en instancias siguientes, `collection.count() > 0` evita reprocesar.

Ejemplo con una pregunta temática puntual -- estimación de densidad de kernel (KDE), cubierta en la Unidad 1 (Estadística Descriptiva):

In [5]:
pregunta = "¿Qué es la estimación de densidad de kernel (KDE) y en qué se diferencia de un histograma?"

respuesta = tutor.ask(pregunta)
print(respuesta)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


¡Hola! Es una excelente pregunta que nos permite profundizar en dos métodos fundamentales para la estimación de la densidad de una distribución de datos, especialmente relevante en el análisis de datos de caracterización nanotecnológica.

Como tu Agente Tutor experto en Probabilidad y Estadística Inferencial, te guiaré paso a paso para comprender la Estimación de Densidad por Kernel (KDE) y sus diferencias con un histograma.

---

### ¿Qué es la Estimación de Densidad por Kernel (KDE)?

La **Estimación No Paramétrica de Densidad por Kernel (KDE)** es una técnica utilizada para estimar la función de densidad de probabilidad (PDF) de una variable aleatoria, basándose en una muestra de datos. La característica "no paramétrica" significa que no asumimos una forma específica (como una distribución normal, exponencial, etc.) para la distribución subyacente de los datos.

En el contexto de la Ingeniería en Nanotecnología, por ejemplo, cuando analizamos el diámetro de nanopartículas y no quere

---

## Para profundizar

- **Arquitectura completa de agentes**: `README.md` (sección "🏛️ Sistema de Agentes y Gobernanza", con diagrama del flujo entre agentes) y `GOVERNANCE.md`.
- **API detallada de cada agente**: los tests son hoy la fuente más completa de ejemplos de invocación -- `tests/test_orchestrator_agent.py`, `tests/council/test_council_pipeline.py`, `tests/test_stats_tutor_agent.py`.
- **Bibliografía indexada por `StatsTutorAgent`**: ver `bibliografia/README.md`.